# 06 — Transfer thermal stress to the walking network

This notebook creates the central methodological bridge between the thermal raster and accessibility analysis.

For each walking edge inside the thermal raster footprint:

1. sample UTCI at the edge midpoint;
2. compute normal walking time;
3. calculate heat excess above 32 °C;
4. convert that stress into a generalized walking cost.

The PoC cost function is:

\[
c_e^{heat}=t_e[1+\lambda\max(UTCI_e-32,0)]
\]

with walking speed = 1.4 m/s and \(\lambda=0.05\).

In [ ]:
from pathlib import Path

# The notebooks_v02 folder is expected to sit directly inside the repository.
# If you open Jupyter from another working directory, set PROJECT_ROOT manually.
HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE.parent if HERE.name == "notebooks_v02" else HERE

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
OUTPUTS = PROJECT_ROOT / "outputs"

for folder in [DATA_RAW, DATA_INTERIM, OUTPUTS]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import rasterio

from shapely.geometry import box

WALKING_SPEED_MPS = 1.4
UTCI_THRESHOLD = 32.0
LAMBDA_HEAT = 0.05

G = ox.load_graphml(DATA_RAW / "sevilla_walk.graphml")
nodes, edges = ox.graph_to_gdfs(G)

nodes = nodes.to_crs("EPSG:25830")
edges = edges.to_crs("EPSG:25830")

# OSMnx needs x/y columns when rebuilding a graph from GeoDataFrames.
nodes["x"] = nodes.geometry.x
nodes["y"] = nodes.geometry.y

## 1. Restrict the walking network to the thermal raster footprint

In [ ]:
utci_path = OUTPUTS / "solweig" / "summary" / "utci_max.tif"

with rasterio.open(utci_path) as src:
    raster_extent = box(*src.bounds)
    print("UTCI CRS:", src.crs)
    print("UTCI bounds:", src.bounds)

edges_poc = edges[edges.intersects(raster_extent)].copy()

node_ids = (
    set(edges_poc.index.get_level_values("u"))
    | set(edges_poc.index.get_level_values("v"))
)

nodes_poc = nodes.loc[nodes.index.intersection(node_ids)].copy()

print("PoC nodes:", len(nodes_poc))
print("PoC edges:", len(edges_poc))

## 2. Sample UTCI at each edge midpoint

In [ ]:
edges_poc["midpoint"] = edges_poc.geometry.interpolate(
    0.5,
    normalized=True,
)

coordinates = [
    (point.x, point.y)
    for point in edges_poc["midpoint"]
]

with rasterio.open(utci_path) as src:
    edges_poc["utci_max"] = [
        value[0] for value in src.sample(coordinates)
    ]

edges_poc.loc[
    ~np.isfinite(edges_poc["utci_max"]),
    "utci_max",
] = np.nan

display(edges_poc["utci_max"].describe())

## 3. Build normal and thermal generalized costs

In [ ]:
edges_poc["walk_time_s"] = (
    edges_poc["length"] / WALKING_SPEED_MPS
)

edges_poc["heat_excess"] = np.maximum(
    edges_poc["utci_max"] - UTCI_THRESHOLD,
    0,
)

edges_poc["thermal_cost_s"] = (
    edges_poc["walk_time_s"]
    * (
        1
        + LAMBDA_HEAT
        * edges_poc["heat_excess"]
    )
)

# This ratio is the multiplicative thermal penalty, not normalized UTCI.
edges_poc["cost_ratio"] = (
    edges_poc["thermal_cost_s"]
    / edges_poc["walk_time_s"]
)

display(
    edges_poc[
        ["utci_max", "walk_time_s", "thermal_cost_s", "cost_ratio"]
    ].describe()
)

## 4. Rebuild and save the thermalized network

In [ ]:
G_thermal = ox.graph_from_gdfs(
    nodes_poc,
    edges_poc,
    graph_attrs={"crs": "EPSG:25830"},
)

thermal_graph_path = DATA_INTERIM / "sevilla_walk_thermal.graphml"

ox.save_graphml(
    G_thermal,
    filepath=thermal_graph_path,
)

print("Saved:", thermal_graph_path)
print("Nodes:", G_thermal.number_of_nodes())
print("Edges:", G_thermal.number_of_edges())

## 5. Visualize the heat-adjusted cost multiplier

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

edges_poc.plot(
    ax=ax,
    column="cost_ratio",
    linewidth=1.4,
    legend=True,
)

ax.set_title("Heat-adjusted walking cost ratio")
ax.set_axis_off()
plt.tight_layout()
plt.show()